# Notebook 21 — Virtual envs & dependency pins

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `20-positional-keyword-only-signatures.ipynb`

**Next up:** `22-html-parser-text.ipynb`

---

## Learning objectives

- Explain **`venv`** isolation vs global interpreters.
- Parse **`requirements.txt`**-style pins mentally.
- Know why reproducible stacks beat notebook-only installs.

## Table of contents

1. **`python -m venv` recap**
2. **`pip` introspection**
3. **Pin notation (`==`, `>=`)**
4. **Progressive drills — freeze lists → comments → editable installs mindset**
5. **Exercise — `parse_pins`**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · `python -m venv` recap

*Explanation:* Creates **`Scripts/python`** (Windows) / **`bin/python`** (POSIX) isolated — protects system Python.


```bash
python -m venv .venv
# Windows activation:
.venv\Scripts\activate
# POSIX:
source .venv/bin/activate
python -m pip install --upgrade pip
```


## 2 · `pip` introspection

*Explanation:* **`pip freeze`** snapshots pins — CI compares against locked requirements.


In [ ]:
import subprocess

proc = subprocess.run(
    ["python", "-m", "pip", "--version"],
    capture_output=True,
    text=True,
    check=True,
)
print(proc.stdout.strip())


## 3 · Pin notation

*Explanation:* **`==`** pins exact releases; **`>=`** expresses floors — combine judiciously before **`torch`**-scale stacks.


In [ ]:
pins = ["numpy>=1.24", "httpx==0.27.0"]
for pin in pins:
    print(pin.split("==")[0] if "==" in pin else pin.split(">=")[0])


---

## Progressive drills — **easy → harder**

**Shipping demos** relies on reproducible installs — drills mimic CI parsing guards.


### A · Easiest — strip inline comments

Comments belong outside resolver graphs.


In [ ]:
raw = "numpy>=1.24  # cuda build\nhttpx==0.27.0"
clean_lines = [ln.split("#")[0].strip() for ln in raw.splitlines() if ln.strip()]
print(clean_lines)


### B · Medium — normalize casing

Package names conventionally lowercase — normalize before hashing locks.


In [ ]:
specs = {"NumPy": "1.26.0", "HTTPCORE": "1.0.0"}
canonical = {k.lower(): v for k, v in specs.items()}
print(canonical)


### C · Harder — editable install vocabulary

**`-e .`** wires local packages — priceless monorepo ergonomics.


```bash
pip install -e .
```

Adds **`pyproject.toml`** projects onto **`PYTHONPATH`** without reinstall loops.


### Exercise — `parse_pins`

Implement **`parse_pins(blob: str) -> dict[str, str]`** scanning **`blob.splitlines()`**. Skip blanks after stripping. Strip inline **`#`** comments per line. For remaining lines containing **`==`** or **`>=`**, split once on that delimiter (prefer **`==`** when both appear — uncommon). Keys are **`pkg.strip().lower()`**, values are **`version.strip()`**. Ignore malformed lines quietly.


In [ ]:
def parse_pins(blob: str) -> dict[str, str]:
    raise NotImplementedError


sample = """numpy>=1.24 # tensors
httpx==0.27.0

badline
Torch>=2.2  # intentional uppercase vendor spelling for drill
"""
parsed = parse_pins(sample)
assert parsed["numpy"] == "1.24"
assert parsed["httpx"] == "0.27.0"
assert parsed["torch"] == "2.2"
print("OK")


### Solution — parse_pins

<details>
<summary>Click to expand</summary>

```python
def parse_pins(blob: str) -> dict[str, str]:
    out: dict[str, str] = {}
    for raw in blob.splitlines():
        line = raw.split("#")[0].strip()
        if not line:
            continue
        if "==" in line:
            sep = "=="
        elif ">=" in line:
            sep = ">="
        else:
            continue
        pkg, _, ver = line.partition(sep)
        pkg = pkg.strip()
        ver = ver.strip()
        if not pkg or not ver:
            continue
        out[pkg.lower()] = ver
    return out
```

</details>
